# Joshi Part 7: Payoffs & Monte Carlo (Python)

Python implementation of the PayOff classes and Simple Monte Carlo from
*The Concepts and Practice of Mathematical Finance* by Mark S. Joshi.

## Topics
- PayOff class hierarchy (Call, Put, Digital, DoubleDigital)
- Simple Monte Carlo pricing for European options
- Convergence table
- Anti-thetic variance reduction

In [ ]:
import numpy as np
from abc import ABC, abstractmethod
from dataclasses import dataclass
import math

## 1. PayOff Classes

Joshi uses C++ virtual inheritance for polymorphic payoffs.
In Python we use abstract base classes.

In [ ]:
class PayOff(ABC):
    """Abstract payoff function."""
    @abstractmethod
    def __call__(self, spot: float) -> float:
        ...

@dataclass
class Call(PayOff):
    strike: float
    def __call__(self, spot: float) -> float:
        return max(spot - self.strike, 0.0)

@dataclass
class Put(PayOff):
    strike: float
    def __call__(self, spot: float) -> float:
        return max(self.strike - spot, 0.0)

@dataclass
class DigitalCall(PayOff):
    strike: float
    def __call__(self, spot: float) -> float:
        return 1.0 if spot > self.strike else 0.0

@dataclass
class DigitalPut(PayOff):
    strike: float
    def __call__(self, spot: float) -> float:
        return 1.0 if spot < self.strike else 0.0

@dataclass
class DoubleDigital(PayOff):
    lower: float
    upper: float
    def __call__(self, spot: float) -> float:
        return 1.0 if self.lower < spot < self.upper else 0.0

@dataclass
class Straddle(PayOff):
    strike: float
    def __call__(self, spot: float) -> float:
        return abs(spot - self.strike)

# Test payoffs
call = Call(100)
put = Put(100)
dc = DigitalCall(100)

for spot in [90, 100, 110]:
    print(f"S={spot}: Call={call(spot):.1f}, Put={put(spot):.1f}, DigitalCall={dc(spot):.1f}")

## 2. Simple Monte Carlo

Simulate GBM paths: $S(T) = S(0) \exp\left((r - \frac{\sigma^2}{2})T + \sigma\sqrt{T}Z\right)$

where $Z \sim N(0,1)$.

In [ ]:
def simple_monte_carlo(payoff, spot, rate, vol, expiry, num_paths, seed=42):
    """Simple Monte Carlo European option pricer."""
    rng = np.random.default_rng(seed)
    drift = (rate - 0.5 * vol**2) * expiry
    vol_sqrt_t = vol * math.sqrt(expiry)
    discount = math.exp(-rate * expiry)
    
    z = rng.standard_normal(num_paths)
    spot_t = spot * np.exp(drift + vol_sqrt_t * z)
    payoffs = np.array([payoff(s) for s in spot_t])
    
    mean = payoffs.mean()
    std_error = payoffs.std(ddof=1) / math.sqrt(num_paths)
    
    price = discount * mean
    se = discount * std_error
    return price, se

# Price ATM call: S=100, K=100, r=5%, vol=20%, T=1
call = Call(100)
price, se = simple_monte_carlo(call, 100, 0.05, 0.2, 1.0, 200_000)
print(f"MC Call Price: {price:.4f} ± {se:.4f}")
print(f"BS Exact:      10.4506")
print(f"95% CI:        [{price - 1.96*se:.4f}, {price + 1.96*se:.4f}]")

## 3. Convergence Table

Track the MC estimate at powers of 2 to observe convergence.

In [ ]:
def monte_carlo_convergence(payoff, spot, rate, vol, expiry, max_paths, seed=42):
    """MC with convergence table at powers of 2."""
    rng = np.random.default_rng(seed)
    drift = (rate - 0.5 * vol**2) * expiry
    vol_sqrt_t = vol * math.sqrt(expiry)
    discount = math.exp(-rate * expiry)
    
    results = []
    running_sum = 0.0
    running_sum_sq = 0.0
    next_record = 2
    
    for i in range(1, max_paths + 1):
        z = rng.standard_normal()
        spot_t = spot * math.exp(drift + vol_sqrt_t * z)
        pv = payoff(spot_t)
        running_sum += pv
        running_sum_sq += pv * pv
        
        if i == next_record:
            mean = running_sum / i
            var = (running_sum_sq / i - mean**2) * i / (i - 1)
            se = math.sqrt(var / i)
            results.append((i, discount * mean, discount * se))
            next_record *= 2
    
    return results

call = Call(100)
table = monte_carlo_convergence(call, 100, 0.05, 0.2, 1.0, 2**18)

print(f"{'Paths':>10} {'Mean':>14} {'Std Error':>14}")
print("-" * 40)
for paths, mean, se in table:
    print(f"{paths:>10} {mean:>14.6f} {se:>14.6f}")

## 4. Anti-thetic Variance Reduction

For each $Z$, also use $-Z$. The errors partially cancel.

In [ ]:
def monte_carlo_antithetic(payoff, spot, rate, vol, expiry, num_paths, seed=42):
    rng = np.random.default_rng(seed)
    drift = (rate - 0.5 * vol**2) * expiry
    vol_sqrt_t = vol * math.sqrt(expiry)
    discount = math.exp(-rate * expiry)
    
    half = num_paths // 2
    z = rng.standard_normal(half)
    
    spot_up = spot * np.exp(drift + vol_sqrt_t * z)
    spot_down = spot * np.exp(drift - vol_sqrt_t * z)
    
    payoffs = 0.5 * (np.array([payoff(s) for s in spot_up]) + np.array([payoff(s) for s in spot_down]))
    
    mean = payoffs.mean()
    se = payoffs.std(ddof=1) / math.sqrt(half)
    return discount * mean, discount * se

call = Call(100)

price_basic, se_basic = simple_monte_carlo(call, 100, 0.05, 0.2, 1.0, 100_000)
price_anti, se_anti = monte_carlo_antithetic(call, 100, 0.05, 0.2, 1.0, 100_000)

print(f"Basic MC:     {price_basic:.4f} ± {se_basic:.4f}")
print(f"Anti-thetic:  {price_anti:.4f} ± {se_anti:.4f}")
print(f"Variance reduction: {se_basic/se_anti:.1f}x")